In [ ]:
from flask import Flask, request, send_file
from datetime import datetime
import os

import firebase_admin
from firebase_admin import credentials
from firebase_admin import db

app = Flask(__name__)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

VIDEO_PATH = os.path.join(BASE_DIR, "sample.mp4")

cred = credentials.Certificate(os.path.join(BASE_DIR, "serviceAccountKey.json"))

if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {
        "databaseURL": "https://smart-traffic-monitor-789fc-default-rtdb.asia-southeast1.firebasedatabase.app"
    })

traffic_ref = db.reference("traffic_logs")


def get_client_ip():
    x_forwarded_for = request.headers.get("X-Forwarded-For")
    x_real_ip = request.headers.get("X-Real-IP")
    forwarded = request.headers.get("Forwarded")

    if x_forwarded_for:
        return x_forwarded_for.split(",")[0].strip()

    if x_real_ip:
        return x_real_ip

    if forwarded:
        return forwarded

    return request.remote_addr

def save_traffic_log(request_type, data_size_bytes):
    now = datetime.now()

    ip = get_client_ip()
    remote_addr = request.remote_addr
    x_forwarded_for = request.headers.get("X-Forwarded-For")
    x_real_ip = request.headers.get("X-Real-IP")
    forwarded = request.headers.get("Forwarded")
    user_agent = request.headers.get("User-Agent")

    data = {
        "time": str(now),
        "ip": ip,
        "remote_addr": remote_addr,
        "x_forwarded_for": x_forwarded_for,
        "x_real_ip": x_real_ip,
        "forwarded": forwarded,
        "user_agent": user_agent,
        "site_type": "ALLOW_SITE",
        "request_type": request_type,
        "data_size_bytes": data_size_bytes,
        "data_size_mb": round(data_size_bytes / 1024 / 1024, 2),
        "status": "ALLOW"
    }

    traffic_ref.push(data)

    print("Firebase 실제 접속 로그 저장 완료:", data)


@app.route("/")
def home():
    save_traffic_log(
        request_type="HOME",
        data_size_bytes=0
    )

    return """
    <html>
    <head>
        <title>허용 사이트</title>
    </head>

    <body>
        <h1>허용 사이트</h1>
        <p>이 사이트는 보호자 설정에 의해 허용된 사이트입니다.</p>
        <p>접속 시 실제 접속 로그가 Firebase에 저장됩니다.</p>

        <hr>

        <h2>영상 재생 테스트</h2>
        <p>영상을 재생하면 VIDEO 요청 로그가 Firebase에 저장됩니다.</p>

        <video width="640" controls>
            <source src="/video" type="video/mp4">
            브라우저가 video 태그를 지원하지 않습니다.
        </video>

        <br><br>

        <button onclick="location.reload()">
            새로고침 테스트
        </button>
    </body>
    </html>
    """


# 영상 요청 처리
@app.route("/video")
def video():
    if not os.path.exists(VIDEO_PATH):
        return "sample.mp4 파일이 없습니다.", 404

    video_size = os.path.getsize(VIDEO_PATH)

    save_traffic_log(
        request_type="VIDEO",
        data_size_bytes=video_size
    )

    print("영상 요청 발생:", datetime.now(), get_client_ip(), video_size, "bytes")

    return send_file(VIDEO_PATH, mimetype="video/mp4")


# 서버 실행
if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=True
    )